In [1]:
# ==============================================================================
# FABRIC NOTEBOOK 01: BRONZE INGESTION (FRED MACRO + SEC EDGAR + SYNTHETIC APEX)
# ==============================================================================
import requests
import pandas as pd
from pyspark.sql import functions as F

# ------------------------------------------------------------------------------
# 1. INGEST FRED MACROECONOMIC SERIES (Public CSV Endpoints - Zero API Key Needed)
# ------------------------------------------------------------------------------
fred_series = {
    "FEDFUNDS": "Effective Federal Funds Rate",
    "CPIAUCSL": "Consumer Price Index (All Urban)",
    "DGS10": "10-Year Treasury Constant Maturity",
    "DGS2": "2-Year Treasury Constant Maturity",
    "UNRATE": "Unemployment Rate"
}

fred_frames = []
for series_id, desc in fred_series.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df_temp = pd.read_csv(url)
    df_temp.columns = ["observation_date", "raw_value"]
    df_temp["series_id"] = series_id
    df_temp["description"] = desc
    df_temp["value"] = pd.to_numeric(df_temp["raw_value"], errors="coerce")
    fred_frames.append(df_temp[["observation_date", "series_id", "description", "value"]])

pdf_fred = pd.concat(fred_frames, ignore_index=True).dropna(subset=["value"])
pdf_fred = pdf_fred[pdf_fred["observation_date"] >= "2018-01-01"]

sdf_fred = spark.createDataFrame(pdf_fred) \
    .withColumn("observation_date", F.to_date("observation_date")) \
    .withColumn("ingested_at", F.current_timestamp())

sdf_fred.write.format("delta").mode("overwrite").saveAsTable("bronze_fred_macro")
print(f"Loaded {sdf_fred.count():,} rows into bronze_fred_macro")

# ------------------------------------------------------------------------------
# 2. INGEST SEC EDGAR XBRL COMPANY FACTS (Peers: PayPal, Block, Fiserv)
# ------------------------------------------------------------------------------
# SEC requires a custom User-Agent header in the format: "Sample Company Name AdminContact@<sample company domain>.com"
headers = {"User-Agent": "ApexFinAnalytics Research@apexfin-demo.com"}

peers = {
    "PYPL": "0001633917",  # PayPal Holdings
    "XYZ":  "0001512673",  # Block, Inc. (Square)
    "FI":   "0000798354"   # Fiserv, Inc.
}

target_tags = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "OperatingIncomeLoss", "NetIncomeLoss"]
sec_records = []

for ticker, cik in peers.items():
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=headers)
    if resp.status_code == 200:
        us_gaap = resp.json().get("facts", {}).get("us-gaap", {})
        for tag in target_tags:
            if tag in us_gaap:
                units = us_gaap[tag].get("units", {}).get("USD", [])
                for entry in units:
                    if entry.get("form") in ["10-K", "10-Q"] and entry.get("fy", 0) >= 2020:
                        sec_records.append({
                            "ticker": ticker,
                            "cik": cik,
                            "metric_tag": tag,
                            "fiscal_year": int(entry.get("fy")),
                            "fiscal_period": str(entry.get("fp")),
                            "end_date": str(entry.get("end")),
                            "filed_date": str(entry.get("filed")),
                            "val_usd": float(entry.get("val"))
                        })

pdf_sec = pd.DataFrame(sec_records).drop_duplicates(subset=["ticker", "metric_tag", "fiscal_year", "fiscal_period", "end_date"])
sdf_sec = spark.createDataFrame(pdf_sec) \
    .withColumn("end_date", F.to_date("end_date")) \
    .withColumn("filed_date", F.to_date("filed_date")) \
    .withColumn("ingested_at", F.current_timestamp())

sdf_sec.write.format("delta").mode("overwrite").saveAsTable("bronze_sec_peers")
print(f"Loaded {sdf_sec.count():,} rows into bronze_sec_peers")

StatementMeta(, 57de534d-9475-4abf-8d57-26bfeeff1f46, 3, Finished, Available, Finished, False)

Loaded 4,744 rows into bronze_fred_macro
Loaded 618 rows into bronze_sec_peers
